In [ ]:
# Cell 1: Install dependencies (NO torch, NO sentence-transformers)
# chromadb handles embeddings internally via onnxruntime
!pip install -q chromadb groq gradio

In [ ]:
# Cell 2: Add your Groq API key
# Get a FREE key at: https://console.groq.com (takes 30 seconds, no credit card)

GROQ_API_KEY = "your_groq_api_key_here"  # <-- paste your key here

print('✅ Key set') if GROQ_API_KEY != 'your_groq_api_key_here' else print('⚠️  Paste your Groq API key above first!')

In [ ]:
# Cell 3: Load Contentstack Personalize docs
docs_text = [
    "Contentstack Personalize Management API manages Attributes, Audiences, Events, and Experiences. Base URL: https://personalize-api.contentstack.com. Authentication: OAuth Bearer token against authorization header, Project UID against x-project-uid header.",
    "Personalize Attributes are user properties used to build Audience rules. Types: STRING, NUMBER, BOOLEAN. Create: POST /attributes. Presets: COUNTRY, DEVICE_TYPE, SESSION_COUNT, PAGE_VIEW_COUNT.",
    "Personalize Audiences are groups of users. Create: POST /audiences with RuleCombination definition. Operators: EQUALS, GREATER_THAN, LESS_THAN, CONTAINS. Example: US visitors with order value over 1000.",
    "Personalize Experiences create content variations. Types: SEGMENTED and AB_TEST. Create: POST /experiences. Variants have shortUid used with CDA. AB_TEST randomly splits users.",
    "Personalize Events: Impressions and Conversions. Track via Edge API: POST https://personalize-edge.contentstack.com/events. Headers: x-project-uid, x-cs-personalize-user-uid. Impression: type IMPRESSION. Conversion: eventKey + type EVENT.",
    "Personalize Edge API Base URL: https://personalize-edge.contentstack.com. Get Manifest: GET /manifest. Set User Attributes: PUT /user-attributes. Drives audience matching in real time.",
    "JavaScript Personalize Edge SDK: npm i @contentstack/personalize-edge-sdk. Init: await Personalize.init(projectUid). Set attrs: personalizeSdk.set({age:20}). getExperiences(). getVariantAliases() for CDA. triggerImpression(). triggerEvent().",
    "Personalize integration flow: 1.Init SDK 2.Set user attributes 3.getVariantAliases() 4.Pass to CDA as cs-personalize-variant-aliases header 5.CDA returns personalized entry 6.Track impressions 7.Track conversions. User persisted via cs-personalize-user-uid cookie.",
]

print(f'✅ Loaded {len(docs_text)} Personalize doc sections')

In [ ]:
# Cell 4: Chunk + embed into ChromaDB
# chromadb uses its own built-in onnx embeddings (all-MiniLM-L6-v2) - no torch needed
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.Client()

# Built-in default embedding function - uses onnxruntime internally, zero torch dependency
default_ef = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.create_collection(
    name='personalize-docs',
    embedding_function=default_ef
)

# Chunk docs manually (simple split by sentence length)
chunks = []
for i, doc in enumerate(docs_text):
    words = doc.split('. ')
    chunk = ''
    chunk_id = 0
    for w in words:
        if len(chunk) + len(w) < 200:
            chunk += w + '. '
        else:
            if chunk:
                chunks.append({'id': f'{i}_{chunk_id}', 'text': chunk.strip()})
                chunk_id += 1
            chunk = w + '. '
    if chunk:
        chunks.append({'id': f'{i}_{chunk_id}', 'text': chunk.strip()})

collection.add(
    documents=[c['text'] for c in chunks],
    ids=[c['id'] for c in chunks]
)

print(f'✅ Vector store ready with {collection.count()} chunks')

In [ ]:
# Cell 5: Test retrieval
query = 'How do I create an Audience in Contentstack Personalize?'
results = collection.query(query_texts=[query], n_results=3)

print(f'Query: {query}\n')
for i, doc in enumerate(results['documents'][0]):
    print(f'--- Chunk {i+1} ---')
    print(doc)
    print()

In [ ]:
# Cell 6: RAG pipeline using Groq LLM (llama3-8b, free)
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def rag_answer(question):
    # Retrieve relevant chunks
    results = collection.query(query_texts=[question], n_results=3)
    context = '\n'.join(results['documents'][0])

    # Generate answer via Groq
    response = client.chat.completions.create(
        model='llama3-8b-8192',
        messages=[
            {
                'role': 'system',
                'content': 'You are a helpful assistant for Contentstack Personalize documentation. Answer only using the context provided. If you cannot find the answer in the context, say so clearly.'
            },
            {
                'role': 'user',
                'content': f'Context:\n{context}\n\nQuestion: {question}'
            }
        ],
        max_tokens=300
    )

    answer = response.choices[0].message.content
    return answer, results['documents'][0]

print('✅ RAG pipeline ready!')

In [ ]:
# Cell 7: Test questions
def ask(question):
    answer, sources = rag_answer(question)
    print(f'Q: {question}')
    print(f'A: {answer}\n')

ask('How do I create an Audience?')
ask('What is the difference between SEGMENTED and AB_TEST experiences?')
ask('How do I track a conversion event?')

In [ ]:
# Cell 8: Gradio chatbot UI
import gradio as gr

def answer(question, history):
    response, sources = rag_answer(question)
    source_text = '\n'.join(f'- {s[:80]}...' for s in list(set(sources)))
    return f'{response}\n\n---\nSources:\n{source_text}'

gr.ChatInterface(
    fn=answer,
    title='Contentstack Personalize RAG Assistant',
    description='Ask anything about Contentstack Personalize. Answers grounded in real docs.',
    examples=[
        'How do I create an Audience?',
        'What is the difference between SEGMENTED and AB_TEST?',
        'How do I track a conversion event?',
        'How do I authenticate with the Management API?',
        'What does getVariantAliases() return?',
    ],
    theme=gr.themes.Soft()
).launch(share=True)